In [ ]:
import pandas as pd
import ast

In [57]:
# Combine and process all relevant data into one DataFrame
df_rental = pd.read_csv(r"rea_data\domain\Data\vic_rentals_all_cleaned.csv")
df_bus = pd.read_csv(r"rentals_distances_to_bus_stops.csv")
df_bus.rename(columns={"rental_id":"listing_id", "top_distances":"top_bus_distances"}, inplace=True)
df_train_cbd = pd.read_csv(r"rentals_distances_to_train_stops_and_cbd.csv")
df_train_cbd.rename(columns={"rental_id":"listing_id", "top_distances":"top_train_distances"}, inplace=True)
df_tram = pd.read_csv(r"rentals_distances_to_tram_stops.csv")
df_tram.rename(columns={"rental_id":"listing_id", "top_distances":"top_tram_distances"}, inplace=True)
df_school = pd.read_csv(r"rentals_distances_to_schools.csv")
df_school.rename(columns={"rental_id":"listing_id", "top_distances":"top_school_distances"}, inplace=True)
df = df_rental.merge(df_bus, on="listing_id", how="left")
df = df.merge(df_train_cbd, on="listing_id", how="left")
df = df.merge(df_tram, on="listing_id", how="left")
df = df.merge(df_school, on="listing_id", how="left")
df["top_bus_distances"] = df["top_bus_distances"].str.replace("inf", "None")

for col in ["top_bus_distances", "top_train_distances", "top_tram_distances", "top_school_distances", 
            "top_bus_ids", "top_station_ids", "top_tram_ids", "top_school_ids"]:
    df[col] = df[col].apply(lambda x:ast.literal_eval(x) if pd.notnull(x) else [])

df["closest_bus_distance"] = df["top_bus_distances"].str[0]
df["closest_train_distance"] = df["top_train_distances"].str[0]
df["closest_tram_distance"] = df["top_tram_distances"].str[0]
df["closest_school_distance"] = df["top_school_distances"].str[0]
df["closest_bus_id"] = df["top_bus_ids"].str[0]
df["closest_train_id"] = df["top_station_ids"].str[0]
df["closest_tram_id"] = df["top_tram_ids"].str[0]
df["closest_school_id"] = df["top_school_ids"].str[0]


In [58]:
df.head()

,listing_id,suburb,postcode,weekly_rent,bond,available_date,date_listed,days_listed,bedrooms,bathrooms,...,top_school_distances,top_school_ids,closest_bus_distance,closest_train_distance,closest_tram_distance,closest_school_distance,closest_bus_id,closest_train_id,closest_tram_id,closest_school_id
0,16782629,south kingsville,3015,460.0,1994.0,"Tuesday, 02 September 2025",2025-08-13,27.0,2.0,1.0,...,"[1060.45, 2564.66, 4788.34, 115324.43, 224471.84]","[77, 4788, 1527, 3659, 113]",769.19,1889.09,5684.28,1060.45,40836,15345,17934,77
1,17471867,south kingsville,3015,400.0,1738.0,"Thursday, 27 March 2025",2025-03-06,187.0,2.0,1.0,...,"[4190.38, 39213.27, 41875.79, 43234.16, 46083.64]","[3988, 4788, 113, 3659, 1527]",516.57,2230.27,6025.46,4190.38,40834,15345,17934,3988
2,17721851,south kingsville,3015,795.0,3454.0,"Monday, 15 September 2025",2025-08-19,21.0,3.0,2.0,...,"[1777.42, 23266.69, 23738.09, 24221.5, 25347.84]","[3659, 1527, 1690, 4788, 113]",17496.04,3156.90,6073.35,1777.42,40820,vic:rail:SPT,17934,3659
3,17725855,south kingsville,3015,675.0,2933.0,"Wednesday, 20 August 2025",2025-08-21,19.0,3.0,1.0,...,"[38929.89, 41592.41, 42950.78, 43634.11, 45800...","[4788, 113, 3659, 5278, 1527]",2844.32,1946.90,5742.08,38929.89,9006,15345,17934,4788
4,17745057,south kingsville,3015,450.0,1955.0,"Tuesday, 02 September 2025",2025-09-03,6.0,2.0,1.0,...,"[4109.06, 39131.94, 41794.46, 43152.83, 46002.31]","[3988, 4788, 113, 3659, 1527]",435.24,2148.95,5944.13,4109.06,40834,15345,17934,3988


In [59]:
# Separate labels and features while ignoring less useful features
label = "weekly_rent"
numeric_features = ["bond", "days_listed", "bedrooms", "bathrooms", "carspaces", "photo_count", 
                    "video_count", "floorplans_count", "closest_bus_distance", "closest_train_distance",
                    "closest_tram_distance", "closest_school_distance"]
categorical_features = ["suburb", "postcode", "available_date", "date_listed", "property_type", 
                        "domain_page_id", "property_id", "virtual_tour", "primary_type", "secondary_type", 
                        "agency", "agency_id", "agent_names", "closest_bus_id", "closest_train_id",
                    "closest_tram_id", "closest_school_id"]

In [60]:
# Compute correlation for numeric features
correlations = df[numeric_features + [label]].corr()[label].sort_values(ascending=False)
print(correlations)

weekly_rent                1.000000
bond                       0.563270
bathrooms                  0.388489
bedrooms                   0.264648
photo_count                0.247857
carspaces                  0.168096
floorplans_count           0.147281
video_count                0.044532
days_listed               -0.022018
closest_school_distance   -0.045690
closest_bus_distance      -0.141215
closest_train_distance    -0.146343
closest_tram_distance     -0.178896
Name: weekly_rent, dtype: float64


In [ ]:
# Compute mutual information for numeric features
from sklearn.feature_selection import mutual_info_regression

x_num = df[numeric_features].fillna(0)
y = df[label]

mi_scores_num = mutual_info_regression(x_num, y)
mi_scores_num = pd.Series(mi_scores_num, index=numeric_features).sort_values(ascending=False)

print(mi_scores_num)




bond                       4.095423
closest_tram_distance      0.203316
closest_train_distance     0.169589
bathrooms                  0.161832
bedrooms                   0.151413
closest_bus_distance       0.128275
closest_school_distance    0.082664
photo_count                0.069386
carspaces                  0.058193
days_listed                0.027516
floorplans_count           0.019390
video_count                0.010800
dtype: float64


In [ ]:
# Compute mutual information for categorical features
x_cat = pd.get_dummies(df[categorical_features], drop_first=True)
y = df[label]

x_cat = x_cat.fillna(0)

mi_scores_cat = mutual_info_regression(x_cat, y)
mi_scores_cat = pd.Series(mi_scores_cat, index=x_cat.columns).sort_values(ascending=False)

print(mi_scores_cat)


postcode                                      0.302686
agency_id                                     0.175459
closest_school_id                             0.174709
closest_tram_id                               0.161694
domain_page_id                                0.044354
                                                ...   
agent_names_Daisy Singh, Courtney Jane        0.000000
agent_names_Daisy Singh                       0.000000
agent_names_DJ Kaur                           0.000000
agent_names_Cynthia Mercieca, Amanda Galea    0.000000
closest_train_id_vic:rail:YVE                 0.000000
Length: 13034, dtype: float64
